# 🛡️ Safe RAG: 금융 문서 임베딩 벡터 역전공격 방어 실험

## 실험 개요
RAG(Retrieval-Augmented Generation) 시스템의 벡터 DB에는 실제 금융 문서의 임베딩이 저장된다.
**Vec2Text** 같은 역전공격(embedding inversion attack)은 이 벡터만으로 원문 텍스트를 상당 수준 복원할 수 있어,
고객 PII(개인식별정보) 유출 위험이 존재한다.

이 노트북은 다음 흐름으로 방어 전략의 효과를 실험한다:

```
금융 문서 생성 → 임베딩 → Vec2Text 역전공격 → PCA/PII 방어 → 재공격 → 트레이드오프 분석
```

| 단계 | 내용 |
|------|------|
| Step 1 | 금융 PII 데이터셋 생성 + 임베딩 + FAISS 인덱스 |
| Step 2 | 원본 벡터 역전공격 (공격 기준선 측정) |
| Step 3 | 정규화 벡터 역전공격 (단순 정규화의 방어 효과 확인) |
| Step 4 | PCA 방어 벡터 역전공격 + RAG 트레이드오프 분석 |
| Step 5 | PII-aware 억제 + PCA 복합 방어 |
| Step 6 | 최적 PCA n_components 탐색 |
| Bonus | 공용 PCA 벡터 배포 Feasibility Check |

## Step 1. 금융 문서 임베딩 벡터 생성

### 목적
공격 실험에 사용할 현실적인 금융 문서 데이터셋을 준비한다.
vec2text 모델이 **영어로 학습**되어 있어, 한국어 문서를 그대로 쓰면 공격 효과가 과소평가된다.
따라서 실제 금융권 문서와 유사한 **영문 PII 포함 문서**를 GPT-4o로 생성한다.

### 환경 설정

### 1-0. 환경 설정

작업 디렉토리 확인, OpenAI API 키 로드, 의존 패키지 설치를 순서대로 수행한다.
`.env` 파일에 `OPENAI_API_KEY`가 설정되어 있어야 하며, GPU 환경(Colab A100)을 권장한다.

In [ ]:
! pwd

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # .env 자동 읽기
api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:10], "...")

In [ ]:
! pip install -r requirements.txt

### 1-1. 영문 금융 PII 데이터셋 생성

**목적**: Vec2Text가 영어 모델임을 고려해 한국 금융 문서 구조를 그대로 유지하되 영문으로 작성.

**생성 문서 유형** (10종):
- 대출 심사 보고서, 보험 가입 신청서, 신용카드 이용 내역
- 증권 계좌 개설 서류, 이상 거래 탐지 보고서 등

각 문서에는 실명·SSN·계좌번호·주소 등 **민감 PII 필드**가 포함되며,
이것이 역전공격 시 유출될 수 있는 타깃 정보가 된다.

In [ ]:
import os
import json
from openai import OpenAI
from pathlib import Path
from tqdm import tqdm

client = OpenAI()

# 10가지 문서 유형 정의
DOC_TYPES = [
    {
        "type": "loan_review",
        "title": "Loan Review Report",
        "description": "Credit review report with customer PII, loan amount, interest rate, credit grade, and approval result",
        "pii_fields": ["full name", "SSN", "account number", "loan amount", "interest rate"]
    },
    {
        "type": "fraud_detection",
        "title": "Fraud Transaction Detection Report",
        "description": "Suspicious transaction detection report with customer account, transaction pattern, timestamps, and action taken",
        "pii_fields": ["full name", "account number", "SSN", "transaction amounts", "timestamps"]
    },
    {
        "type": "pb_asset_management",
        "title": "PB Customer Asset Management Record",
        "description": "Private banking asset management record with total assets, portfolio breakdown, and PB advisor info",
        "pii_fields": ["full name", "date of birth", "account number", "total assets", "portfolio allocation"]
    },
    {
        "type": "suspicious_account",
        "title": "Suspicious Account (Money Mule) Report",
        "description": "Money mule / voice phishing related account report with suspect account details and investigation status",
        "pii_fields": ["full name", "account number", "SSN", "report number", "case status"]
    },
    {
        "type": "foreign_remittance",
        "title": "Foreign Remittance Approval Record",
        "description": "International wire transfer approval with sender/receiver details, amount, exchange rate, and approval number",
        "pii_fields": ["full name", "account number", "SSN", "remittance amount", "approval number"]
    },
    {
        "type": "payroll_transfer",
        "title": "Employee Payroll Transfer Record",
        "description": "Employee salary payment record with department, employee ID, base salary, bonus, and net amount",
        "pii_fields": ["full name", "employee ID", "account number", "SSN", "salary details"]
    },
    {
        "type": "mortgage_contract",
        "title": "Real Estate Mortgage Loan Contract",
        "description": "Mortgage loan contract with collateral property details, LTV ratio, loan term, and interest rate",
        "pii_fields": ["full name", "SSN", "account number", "property address", "loan amount"]
    },
    {
        "type": "internal_audit",
        "title": "Internal Audit Violation Report",
        "description": "Internal audit finding report with employee misconduct, affected customer accounts, and disciplinary action",
        "pii_fields": ["employee name", "employee ID", "victim account number", "victim SSN", "violation details"]
    },
    {
        "type": "insurance_claim",
        "title": "Insurance Claim Review Record",
        "description": "Insurance claim adjudication record with policy number, claim reason, hospitalization details, and payout",
        "pii_fields": ["full name", "SSN", "policy number", "claim amount", "payout account"]
    },
    {
        "type": "household_debt",
        "title": "Household Debt Management Summary",
        "description": "Comprehensive household debt management record with DSR ratio, multiple loan details, and risk grade",
        "pii_fields": ["full name", "SSN", "multiple account numbers", "total debt", "DSR ratio"]
    }
]

SYSTEM_PROMPT = """You are a financial document generator for a Korean bank's internal system.
Generate realistic English-language internal financial documents with realistic fake PII data.
Each document must:
1. Include realistic fake personal information (names, SSNs in XXX-XX-XXXX format, account numbers)
2. Include specific financial figures and dates
3. Follow formal internal banking document structure
4. Be 150-250 words long
5. Include a document header with type and reference number
Output ONLY the document text, no explanations."""

def generate_document(doc_type: dict, variation_num: int) -> str:
    prompt = f"""Generate variation #{variation_num} of a {doc_type['title']}.

Document description: {doc_type['description']}
Required PII fields to include: {', '.join(doc_type['pii_fields'])}

Make each variation unique with different:
- Customer names (Western names)
- Financial figures
- Dates (use 2025-2026 dates)
- Specific details relevant to the document type

Output the document directly."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.9,
        max_tokens=400
    )
    return response.choices[0].message.content

def main():
    output_dir = Path("docs_english")
    output_dir.mkdir(exist_ok=True)

    total = len(DOC_TYPES) * 10  # 10가지 유형 × 10개 = 100개
    generated = 0
    metadata = []

    print(f"총 {total}개 영어 금융 문서 생성 시작\n")

    for doc_type in DOC_TYPES:
        print(f"[{doc_type['title']}] 10개 생성 중...")
        for i in tqdm(range(1, 11), desc=doc_type['type']):
            try:
                content = generate_document(doc_type, i)
                filename = f"{doc_type['type']}_{i:02d}.txt"
                filepath = output_dir / filename

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(content)

                metadata.append({
                    "filename": filename,
                    "type": doc_type['type'],
                    "title": doc_type['title'],
                    "variation": i
                })
                generated += 1

            except Exception as e:
                print(f"  오류 ({filename}): {e}")

    # 메타데이터 저장
    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"\n생성 완료: {generated}/{total}개")
    print(f"저장 위치: {output_dir.absolute()}")
    print(f"포함된 PII 유형: 이름, SSN, 계좌번호, 금액, 날짜")

if __name__ == "__main__":
    main()

### 1-2. 문서 로드 및 청크 분할

**목적**: 생성된 금융 문서를 RAG에 적합한 크기로 분할한다.

| 파라미터 | 값 | 이유 |
|----------|-----|------|
| chunk_size | 512 토큰 | Ada-002의 최적 입력 길이 |
| chunk_overlap | 64 토큰 | 문장 경계 정보 손실 방지 |

**인사이트**: 청크가 너무 크면 임베딩이 희석되고, 너무 작으면 맥락이 끊긴다.
512/64 설정은 금융 문서의 단락 구조에 맞게 선택했다.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader("./docs_english/", glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(documents)
print(f"총 청크 수: {len(chunks)}")

### 1-3. 임베딩 생성 및 FAISS 인덱스 구축

**목적**: 각 청크를 `text-embedding-ada-002`로 1536차원 벡터로 변환하고,
FAISS `IndexFlatIP`(내적 기반 유사도)로 저장한다.

- `embeddings_unsafe_raw.npy`: 원본 벡터 (공격 실험용)
- `faiss_unsafe.index`: 정규화 벡터 인덱스 (RAG 검색용)

**인사이트**: FAISS는 내부 저장 시 L2 정규화를 적용하므로,
원본 벡터와 FAISS 내부 벡터는 크기(magnitude)만 다르고 방향은 동일하다.
역전공격 실험에서는 이 두 버전을 분리해 비교한다.

In [ ]:
import numpy as np
import faiss
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

def build_and_save_index(chunks, index_path: str, raw_path: str):
    embeddings_list = []
    for chunk in tqdm(chunks, desc=f"임베딩 생성"):
        embeddings_list.append(get_embedding(chunk.page_content))

    embeddings_raw = np.vstack(embeddings_list)

    np.save(raw_path, embeddings_raw)
    print(f"원본 벡터 저장: {raw_path} | shape: {embeddings_raw.shape}")

    embeddings_norm = embeddings_raw.copy()
    faiss.normalize_L2(embeddings_norm)
    index = faiss.IndexFlatIP(1536)
    index.add(embeddings_norm)
    faiss.write_index(index, index_path)
    print(f"FAISS 인덱스 저장: {index_path} | {index.ntotal}개 벡터")

    return embeddings_raw

embeddings_raw = build_and_save_index(
    chunks,
    index_path="faiss_unsafe.index",
    raw_path="embeddings_unsafe_raw.npy"
)

### 1-4. RAG 기본 성능 측정 (Recall@K)

**목적**: 방어 적용 전 기준 RAG 성능을 먼저 측정한다.
이후 방어를 적용했을 때 성능 저하를 이 값과 비교한다.

**Recall@K**: 특정 청크를 쿼리로 넣었을 때 동일 청크가 상위 K개 결과 안에 나오는 비율.
이상적인 RAG 인덱스라면 Recall@5 = 100%가 되어야 한다.

In [ ]:
def recall_at_k(index_path, chunks, k=5, n_eval=20):
    index = faiss.read_index(index_path)
    hits = 0
    for i in range(min(n_eval, len(chunks))):
        vec = get_embedding(chunks[i].page_content).reshape(1, -1)
        faiss.normalize_L2(vec)
        _, indices = index.search(vec, k)
        if i in indices[0]:
            hits += 1
    score = hits / min(n_eval, len(chunks))
    print(f"Recall@{k} ({index_path}): {score:.2f}")
    return score

recall_at_k("faiss_unsafe.index", chunks)
# 목표: 0.90 이상

### 1-5. FAISS 인덱스에서 정규화 벡터 추출

**목적**: FAISS에 저장된 L2 정규화 벡터를 numpy 배열로 복원한다.
`index.reconstruct_n()`은 FAISS 인덱스 내부의 벡터를 직접 읽어오는 함수다.

**인사이트**: 이 벡터는 원본과 방향이 동일하지만 크기가 1로 고정되어 있다.
Step 3에서 '정규화만으로 방어가 되는가'를 검증하기 위해 별도로 추출한다.

In [ ]:
import faiss
import numpy as np

index = faiss.read_index("faiss_unsafe.index")
n = index.ntotal
dim = index.d

embeddings_norm = np.zeros((n, dim), dtype=np.float32)
index.reconstruct_n(0, n, embeddings_norm)

print(f"추출된 정규화 벡터: {embeddings_norm.shape}")

### 1-6. 임베딩 유틸 함수 재정의

이후 단계에서 새로운 텍스트(예: 쿼리, PII 샘플)를 임베딩할 때 사용하는 공통 함수.
모델은 전 단계와 동일한 `text-embedding-ada-002`를 유지한다.

In [7]:
from openai import OpenAI
import numpy as np

client = OpenAI()  # OPENAI_API_KEY 환경변수 필요

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

### 1-7. PCA 방어 벡터 생성 (1차 시도, n=128)

**목적**: 원본 1536차원 벡터를 128차원으로 압축 후 역투영(reconstruct)한다.
이 과정에서 하위 주성분 정보가 손실되며, 역전공격 모델이 복원에 필요한 세부 패턴이 제거된다.

**원리**:
```
원본 (1536d) → PCA 압축 (128d) → 역투영 (1536d) → 방어 벡터
```
차원은 원본과 동일하게 유지되므로 FAISS 인덱스 구조를 그대로 사용할 수 있다.

**인사이트**: n=128은 1536의 약 8% 수준으로, 상당한 정보 손실이 발생한다.
방어력은 높겠지만 RAG 검색 품질 저하도 클 수 있다. Step 4에서 검증한다.

In [ ]:
from sklearn.decomposition import PCA
import torch
import numpy as np

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
print(f"원본 벡터 shape: {embeddings_raw.shape}")

embeddings_matrix = embeddings_raw.astype(np.float32)
n_samples, n_features = embeddings_matrix.shape

n_components = min(128, n_samples - 1)
print(f"PCA 설정: {n_features}d → {n_components}d → {n_features}d")

pca = PCA(n_components=n_components, svd_solver='full')
pca.fit(embeddings_matrix)

compressed    = pca.transform(embeddings_matrix)
reconstructed = pca.inverse_transform(compressed).astype(np.float32)

np.save("embeddings_defense_pca.npy", reconstructed)

retained = pca.explained_variance_ratio_.sum()
print(f"PCA 방어 벡터 저장 완료")
print(f"보존된 분산: {retained:.1%}")

## Step 2. 원본 벡터로 Vec2Text 역전공격

### 목적
`text-embedding-ada-002`가 생성한 **원본 1536차원 벡터**에 직접 Vec2Text를 적용한다.
이 결과가 **공격 기준선(baseline)**이 된다.

### 측정 지표
- **ROUGE-1 F1**: 복원된 텍스트와 원문의 단어 겹침 비율. 높을수록 공격 성공.

### 예상
정보 손실 없는 원본 벡터이므로 ROUGE가 가장 높게 나올 것.

In [ ]:
import vec2text
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
print(f"원본 벡터 shape: {embeddings_raw.shape}")

sample_vecs = torch.tensor(embeddings_raw[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores_raw = []

for i in tqdm(range(10), desc="원본 벡터 역전공격"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=4,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_raw.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (원본 벡터 기준): {np.mean(scores_raw):.3f}")
print(f"최고: {max(scores_raw):.3f} | 최저: {min(scores_raw):.3f}")
print("="*50)

## Step 3. 정규화 벡터로 Vec2Text 역전공격

### 목적
FAISS는 내부적으로 벡터를 L2 정규화한다. 정규화 자체가 방어 효과를 갖는지 확인한다.

### 인사이트 (가설)
정규화는 벡터의 **방향**은 유지하고 크기만 제거하므로,
의미 정보는 그대로 남아 있어 방어 효과가 거의 없을 것으로 예상한다.
→ 만약 ROUGE가 원본과 비슷하게 나온다면, **정규화만으로는 충분하지 않음**을 의미.

In [ ]:
import vec2text
import torch
import faiss
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")

index = faiss.read_index("faiss_unsafe.index")
n = index.ntotal
dim = index.d
embeddings_norm = np.zeros((n, dim), dtype=np.float32)
index.reconstruct_n(0, n, embeddings_norm)

sample_vecs = torch.tensor(embeddings_norm[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores_norm = []

for i in tqdm(range(10), desc="역전공격 진행"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=0,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_norm.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (Unsafe 베이스라인): {np.mean(scores_norm):.3f}")
print(f"최고: {max(scores_norm):.3f} | 최저: {min(scores_norm):.3f}")
print("="*50)
print("→ 이 숫자가 높을수록 역전공격 성공, 낮을수록 자체 방어력 있음")

## Step 4-A. PCA 방어 벡터의 역전공격 방어 성능 측정

### 목적
128차원으로 압축 후 역투영(reconstruct)한 PCA 방어 벡터가 Vec2Text 공격을 얼마나 막는지 측정한다.

### 원리
PCA 재구성은 하위 주성분(노이즈 성분)을 버리는 효과가 있어,
역전공격 모델이 의존하는 세밀한 텍스트 패턴 정보가 손상된다.

### 인사이트 (가설)
압축률이 높을수록 (n_components가 작을수록) 방어력은 강해지지만,
RAG 검색 품질도 함께 저하되는 **트레이드오프**가 존재할 것.

In [ ]:
import vec2text
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")

embeddings_pca = np.load("embeddings_defense_pca.npy")
print(f"PCA 방어 벡터 shape: {embeddings_pca.shape}")

sample_vecs = torch.tensor(embeddings_pca[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores_pca = []

for i in tqdm(range(10), desc="PCA 방어 벡터 역전공격"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=0,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_pca.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (PCA 방어 벡터): {np.mean(scores_pca):.3f}")
print(f"최고: {max(scores_pca):.3f} | 최저: {min(scores_pca):.3f}")
print("="*50)

## Step 4-B. RAG 검색 정확성 vs 방어 성능 트레이드오프 분석

### 목적
방어가 강해질수록 RAG 성능이 얼마나 희생되는지 정량화한다.
실용적인 Safe RAG의 배포 기준을 도출한다.

### 평가 기준
| 지표 | 의미 | 목표 |
|------|------|------|
| Recall@5 | 정답 청크가 상위 5개 안에 있을 확률 | ≥ 90% |
| ROUGE-1 감소율 | 공격 성공률이 얼마나 줄었는가 | > 20% |

### 인사이트
두 지표를 동시에 만족하는 n_components 값을 찾는 것이 핵심이다.
이 분석 결과가 Step 6(최적 PCA 탐색)의 기준이 된다.

In [ ]:
import numpy as np
import faiss
import matplotlib.pyplot as plt
import pandas as pd

print("="*80)
print("📊 최종 트레이드오프 분석: RAG 정확성 vs 벡터 보안")
print("="*80)

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
embeddings_pca = np.load("embeddings_defense_pca.npy")

attack_results = {
    "원본 벡터": {
        "rouge_mean": np.mean(scores_raw),
        "rouge_std": np.std(scores_raw),
        "rouge_max": max(scores_raw),
        "rouge_min": min(scores_raw)
    },
    "정규화 벡터": {
        "rouge_mean": np.mean(scores_norm),
        "rouge_std": np.std(scores_norm),
        "rouge_max": max(scores_norm),
        "rouge_min": min(scores_norm)
    },
    "PCA 방어 벡터": {
        "rouge_mean": np.mean(scores_pca),
        "rouge_std": np.std(scores_pca),
        "rouge_max": max(scores_pca),
        "rouge_min": min(scores_pca)
    }
}

print("\n🔴 벡터 역전공격 성공도 (ROUGE-1):")
print("-" * 80)
for name, metrics in attack_results.items():
    print(f"{name:20} | 평균: {metrics['rouge_mean']:.3f} ± {metrics['rouge_std']:.3f} | 범위: [{metrics['rouge_min']:.3f}, {metrics['rouge_max']:.3f}]")

def evaluate_rag_recall(embeddings, name, chunks, k=5, n_eval=50):
    n_samples = min(n_eval, len(chunks))
    embeddings_eval = embeddings.copy()
    faiss.normalize_L2(embeddings_eval)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings_eval)

    hits = 0
    retrieval_scores = []

    for i in range(n_samples):
        vec = embeddings[i:i+1].copy()
        faiss.normalize_L2(vec)
        distances, indices = index.search(vec, k)

        if i in indices[0]:
            hits += 1
        retrieval_scores.append(distances[0][0])

    recall_at_k = hits / n_samples
    avg_relevance = np.mean(retrieval_scores)

    return {"recall_at_k": recall_at_k, "avg_relevance": avg_relevance}

rag_results = {}
print("\n🟢 RAG 검색 정확성 (Recall@5):")
print("-" * 80)

for embeddings, name in [
    (embeddings_raw, "원본 벡터"),
    (embeddings_pca, "PCA 방어 벡터")
]:
    result = evaluate_rag_recall(embeddings, name, chunks, k=5, n_eval=50)
    rag_results[name] = result
    print(f"{name:20} | Recall@5: {result['recall_at_k']:.1%} | 평균 관련성: {result['avg_relevance']:.4f}")

print("\n" + "="*80)
print("⚖️  트레이드오프 종합 분석")
print("="*80)

raw_attack = attack_results["원본 벡터"]["rouge_mean"]
pca_attack = attack_results["PCA 방어 벡터"]["rouge_mean"]
attack_reduction = (raw_attack - pca_attack) / raw_attack * 100

raw_recall = rag_results["원본 벡터"]["recall_at_k"]
pca_recall = rag_results["PCA 방어 벡터"]["recall_at_k"]
recall_loss = (raw_recall - pca_recall) / raw_recall * 100

print(f"\n공격 성공률 감소:")
print(f"  → 원본: {raw_attack:.3f} → PCA: {pca_attack:.3f}")
print(f"  → 감소율: {attack_reduction:.1f}%")

print(f"\nRAG 검색 정확성 손실:")
print(f"  → 원본: {raw_recall:.1%} → PCA: {pca_recall:.1%}")
print(f"  → 손실율: {recall_loss:.1f}%")

print(f"\n📈 효율성 지수 (낮을수록 좋음):")
efficiency = attack_reduction / max(recall_loss, 1)
print(f"  → 방어효과 / RAG손실 = {efficiency:.2f}")

print("\n" + "="*80)
if pca_recall >= 0.90 and attack_reduction > 20:
    print("✅ 최적 트레이드오프 달성")
    print(f"   - RAG 성능 유지: {pca_recall:.1%} (목표: 90% 이상)")
    print(f"   - 공격 방어력: {attack_reduction:.1f}% 감소 (좋음)")
elif pca_recall >= 0.90:
    print("⚠️  RAG 성능은 충분하나 방어력 개선 필요")
    print(f"   - 현재: {pca_recall:.1%}, 현재 공격 감소: {attack_reduction:.1f}%")
    print(f"   💡 제안: n_components를 더 줄여 방어력 강화")
else:
    print("⚠️  RAG 성능 목표 미달")
    print(f"   - 현재: {pca_recall:.1%}, 목표: 90% 이상")
    print(f"   💡 제안: n_components를 증가시켜 성능 복구")
print("="*80)

## Step 5. PII-Aware 억제 + PCA 복합 방어

### 목적
PCA의 균일한 압축과 달리, **PII가 집중된 차원을 선별적으로 억제**하는 방법을 시험한다.
이후 PCA와 결합한 복합 방어의 효과를 비교한다.

### PII-Aware 억제 원리
1. PII 포함 청크와 일반 청크의 임베딩 평균을 계산
2. 두 평균의 차이가 큰 차원 = PII 신호가 강한 차원
3. 해당 차원을 0으로 masking → PII 정보 집중 억제

### 인사이트 (가설)
PII-aware 방법은 PCA보다 **타깃이 명확**하지만,
어떤 키워드를 PII로 볼지에 따라 성능이 달라지는 취약점이 있다.

In [ ]:
import numpy as np

def defend_pii_aware(matrix: np.ndarray,
                     pii_samples: np.ndarray,
                     n_suppress: int = 200) -> np.ndarray:
    """
    PII 텍스트 임베딩과 일반 텍스트 임베딩의 차이가 큰 차원을 선택적으로 억제
    pii_samples: PII가 포함된 청크의 임베딩
    """
    pii_mean    = pii_samples.mean(axis=0)
    corpus_mean = matrix.mean(axis=0)
    pii_signal  = np.abs(pii_mean - corpus_mean)

    suppress_dims = np.argsort(pii_signal)[-n_suppress:]

    defended = matrix.copy()
    defended[:, suppress_dims] = 0.0

    norms    = np.linalg.norm(defended, axis=1, keepdims=True)
    defended = (defended / norms).astype(np.float32)

    print(f"PII 집중 차원 {n_suppress}개 억제")
    return defended

embeddings_raw = np.load("embeddings_unsafe_raw.npy")

pii_keywords = ['SSN', 'account', 'loan', 'fraud', 'transaction', 'payment', 'policy', 'salary']
pii_chunk_indices = []

for i, chunk in enumerate(chunks):
    if any(keyword in chunk.page_content for keyword in pii_keywords):
        pii_chunk_indices.append(i)

pii_samples = embeddings_raw[pii_chunk_indices]
print(f"PII 포함 청크 {len(pii_chunk_indices)}개 식별")

embeddings_pii_defended = defend_pii_aware(embeddings_raw, pii_samples, n_suppress=200)
np.save("embeddings_defense_pii.npy", embeddings_pii_defended)
print(f"PII 방어 벡터 저장 완료")

### 5-2. PII-Aware 방어 벡터에 추가 PCA 적용 (복합 방어)

**목적**: PII 차원 억제 후의 벡터에 다시 PCA 재구성을 적용해 이중 방어를 구성한다.

**기대 효과**: PII 억제가 특정 차원의 신호를 제거하고,
PCA가 전체적인 고주파 노이즈 패턴을 제거하므로 두 방법이 **상보적**으로 작용할 것.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

embeddings_pii_defended = np.load("embeddings_defense_pii.npy")
print(f"PII 방어 벡터 로드: {embeddings_pii_defended.shape}")

embeddings_matrix = embeddings_pii_defended.astype(np.float32)
n_samples, n_features = embeddings_matrix.shape

n_components = min(128, n_samples - 1)
print(f"PCA 적용: {n_features}d → {n_components}d → {n_features}d")

pca_combined = PCA(n_components=n_components, svd_solver='full')
pca_combined.fit(embeddings_matrix)

compressed    = pca_combined.transform(embeddings_matrix)
reconstructed = pca_combined.inverse_transform(compressed).astype(np.float32)

norms = np.linalg.norm(reconstructed, axis=1, keepdims=True)
embeddings_combined = (reconstructed / norms).astype(np.float32)

np.save("embeddings_defense_combined.npy", embeddings_combined)

retained = pca_combined.explained_variance_ratio_.sum()
print(f"PCA+PII 복합 방어 벡터 저장 완료")
print(f"보존된 분산: {retained:.1%}")

### 5-3. 복합 방어 벡터(PCA+PII)에 Vec2Text 역전공격

**목적**: PII-Aware 억제 + PCA 복합 방어가 단일 방어보다 얼마나 더 효과적인지 측정한다.

**비교 대상**: 원본, PCA 단독, PII 단독, PCA+PII 복합
→ 복합 방어가 각 단독 방어의 효과를 단순 합산 이상으로 넘는지 확인한다.

In [ ]:
import vec2text
import torch
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")

embeddings_combined = np.load("embeddings_defense_combined.npy")
print(f"PCA+PII 복합 방어 벡터 shape: {embeddings_combined.shape}")

sample_vecs = torch.tensor(embeddings_combined[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

scores_combined = []

for i in tqdm(range(10), desc="PCA+PII 복합 방어 벡터 공격"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=10,
        sequence_beam_width=0,
    )
    rec = result[0]

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores_combined.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (PCA+PII 복합 방어): {np.mean(scores_combined):.3f}")
print(f"최고: {max(scores_combined):.3f} | 최저: {min(scores_combined):.3f}")
print("="*50)

### 5-4. 최종 성능 비교: 모든 방어 전략 종합

**목적**: 지금까지 실험한 모든 방어 전략의 ROUGE-1과 Recall@5를 한 표로 정리한다.

| 방어 전략 | 예상 ROUGE | 예상 Recall@5 |
|-----------|-----------|---------------|
| 원본 벡터 | 최고 (기준선) | 최고 |
| 정규화 벡터 | 기준선과 유사 | 기준선과 유사 |
| PCA 단독 | 감소 | 약간 감소 |
| PII 억제 단독 | 감소 | 약간 감소 |
| PCA + PII 복합 | 가장 낮음 | 가장 낮음 |

이 비교를 통해 **실용 배포에 가장 적합한 방어 전략**을 결정한다.

In [ ]:
import numpy as np
import faiss

print("="*90)
print("🎯 최종 성능 비교: PCA vs PII Guard vs PCA+PII 복합 방어")
print("="*90)

attack_results_final = {
    "원본 벡터": {
        "rouge_mean": np.mean(scores_raw),
        "rouge_std": np.std(scores_raw)
    },
    "PCA 방어": {
        "rouge_mean": np.mean(scores_pca),
        "rouge_std": np.std(scores_pca)
    },
    "PCA+PII 복합": {
        "rouge_mean": np.mean(scores_combined),
        "rouge_std": np.std(scores_combined)
    }
}

print("\n🔴 벡터 역전공격 성공도 비교 (ROUGE-1, 낮을수록 좋음):")
print("-" * 90)
print(f"{'방어 방식':20} | {'평균 ROUGE':15} | {'감소율':15} | {'상태':20}")
print("-" * 90)

raw_baseline = attack_results_final["원본 벡터"]["rouge_mean"]

for defense_name, metrics in attack_results_final.items():
    rouge_mean = metrics["rouge_mean"]
    reduction = (raw_baseline - rouge_mean) / raw_baseline * 100

    if defense_name == "원본 벡터":
        print(f"{defense_name:20} | {rouge_mean:14.3f} | {'기준':14} | {'✗ 취약':20}")
    else:
        status = "✅ 우수" if reduction > 20 else "⚠️  보통"
        print(f"{defense_name:20} | {rouge_mean:14.3f} | {reduction:13.1f}% | {status:20}")

def evaluate_rag_for_defense(embeddings, name):
    n_eval = 50
    embeddings_eval = embeddings.copy()
    faiss.normalize_L2(embeddings_eval)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings_eval)

    hits = 0
    for i in range(min(n_eval, len(chunks))):
        vec = embeddings[i:i+1].copy()
        faiss.normalize_L2(vec)
        distances, indices = index.search(vec, 5)
        if i in indices[0]:
            hits += 1

    return hits / min(n_eval, len(chunks))

print("\n🟢 RAG 검색 정확성 (Recall@5, 높을수록 좋음):")
print("-" * 90)

embeddings_raw_full = np.load("embeddings_unsafe_raw.npy")
embeddings_pca_full = np.load("embeddings_defense_pca.npy")
embeddings_combined_full = np.load("embeddings_defense_combined.npy")

rag_results_final = {
    "원본 벡터": evaluate_rag_for_defense(embeddings_raw_full, "원본"),
    "PCA 방어": evaluate_rag_for_defense(embeddings_pca_full, "PCA"),
    "PCA+PII 복합": evaluate_rag_for_defense(embeddings_combined_full, "Combined")
}

for name, recall in rag_results_final.items():
    print(f"{name:20} | Recall@5: {recall:6.1%}")

print("\n" + "="*90)
print("⚖️  종합 트레이드오프 분석")
print("="*90)

print(f"\n공격 방어 성능 (ROUGE 감소율):")
print(f"  PCA 방어:      {(raw_baseline - attack_results_final['PCA 방어']['rouge_mean']) / raw_baseline * 100:.1f}%")
print(f"  PCA+PII 복합:  {(raw_baseline - attack_results_final['PCA+PII 복합']['rouge_mean']) / raw_baseline * 100:.1f}%")

print(f"\nRAG 정확성 손실:")
baseline_recall = rag_results_final["원본 벡터"]
print(f"  PCA 방어:      {(baseline_recall - rag_results_final['PCA 방어']) / baseline_recall * 100:+.1f}%")
print(f"  PCA+PII 복합:  {(baseline_recall - rag_results_final['PCA+PII 복합']) / baseline_recall * 100:+.1f}%")

print(f"\n" + "="*90)
pca_attack_reduction = (raw_baseline - attack_results_final['PCA 방어']['rouge_mean']) / raw_baseline * 100
combined_attack_reduction = (raw_baseline - attack_results_final['PCA+PII 복합']['rouge_mean']) / raw_baseline * 100
improvement = combined_attack_reduction - pca_attack_reduction

if improvement >= 20:
    print(f"✅ 목표 달성! PCA+PII 복합 방어가 PCA 방어 대비 {improvement:.1f}% 추가 방어")
    print(f"   - PCA 방어:     {pca_attack_reduction:.1f}% 감소")
    print(f"   - PCA+PII 복합: {combined_attack_reduction:.1f}% 감소")
    print(f"   - RAG 성능:     {rag_results_final['PCA+PII 복합']:.1%} (목표 90% 이상)")
elif improvement > 0:
    print(f"⚠️  부분 달성. PCA+PII 복합이 {improvement:.1f}% 추가 방어 제공")
    print(f"   💡 n_suppress 값을 조정하여 추가 성능 향상 가능")
else:
    print(f"⚠️  PCA+PII 복합 방어가 PCA보다 성능이 낮음")
    print(f"   💡 n_suppress 값 감소 권장")
print("="*90)

## Step 6. 최적 PCA n_components 탐색

### 목적
지금까지는 n_components=128로 고정했다.
이 단계에서는 `[512, 384, 256, 192, 128, 96, 64, 32]`를 모두 시험해
**RAG 성능 ≥ 90% 조건 하에서 방어율이 가장 높은 최적값**을 찾는다.

### 왜 이 범위인가?
- `text-embedding-ada-002`의 원본 차원: **1536**
- 512 이상: 분산 대부분 보존 → 방어 효과 미미 예상
- 32 이하: 과도한 정보 손실 → RAG 성능 급락 예상
- 64~256 구간에서 최적값이 존재할 가능성이 높다.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import os

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
print(f"원본 벡터 로드: {embeddings_raw.shape}")

candidates = [512, 384, 256, 192, 128, 96, 64, 32]
output_dir = "pca_candidates"
os.makedirs(output_dir, exist_ok=True)

n_samples, n_features = embeddings_raw.shape

for n_components in candidates:
    if n_components >= n_samples:
        print(f"⏭️  n_components={n_components} 건너뜀 (샘플 수 {n_samples}보다 큼)")
        continue

    pca = PCA(n_components=n_components, svd_solver='full')
    pca.fit(embeddings_raw)

    compressed = pca.transform(embeddings_raw)
    reconstructed = pca.inverse_transform(compressed).astype(np.float32)

    norms = np.linalg.norm(reconstructed, axis=1, keepdims=True)
    normalized = (reconstructed / norms).astype(np.float32)

    np.save(f"{output_dir}/embeddings_n{n_components}.npy", normalized)
    np.save(f"{output_dir}/components_n{n_components}.npy", pca.components_)

    variance_retained = pca.explained_variance_ratio_.sum()
    print(f"n={n_components:3d} | 분산 보존: {variance_retained:.1%} | 벡터 저장 완료")

print(f"\n✅ {len([c for c in candidates if c < n_samples])}개 PCA 후보 벡터 저장 완료")

### 6-2. 각 PCA 후보에 Vec2Text 공격 + Recall@5 측정

**목적**: 32~512 범위의 8개 n_components 후보 각각에 대해
역전공격 ROUGE-1과 RAG Recall@5를 동시에 측정한다.

**소요 시간 주의**: Vec2Text 공격(num_steps=20)이 각 후보당 약 2~5분 소요.
전체 8개 후보 실행 시 GPU에서 약 20~40분 예상.

**결과는 `results.csv`로 저장**되며 다음 셀에서 최적값을 선택한다.

In [ ]:
import vec2text
import torch
import faiss
import numpy as np
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}\n")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

candidates = [512, 384, 256, 192, 128, 96, 64, 32]
output_dir = "pca_candidates"
n_samples = 284

embeddings_raw = np.load("embeddings_unsafe_raw.npy")
originals = [chunks[i].page_content for i in range(10)]

results_list = []

for n_components in candidates:
    if n_components >= n_samples:
        print(f"⏭️  n={n_components} 건너뜀\n")
        continue

    embeddings_candidate = np.load(f"{output_dir}/embeddings_n{n_components}.npy")

    sample_vecs = torch.tensor(embeddings_candidate[:10]).to(device)
    scores = []

    print(f"n={n_components:3d} Vec2Text 공격 중...")
    for i in tqdm(range(10), desc=f"n={n_components}"):
        vec = sample_vecs[i:i+1]
        result = vec2text.invert_embeddings(
            embeddings=vec,
            corrector=corrector,
            num_steps=20,
            sequence_beam_width=0,
        )
        rec = result[0]

        score = scorer.score(originals[i], rec)
        rouge = score['rouge1'].fmeasure
        scores.append(rouge)

    rouge_mean = np.mean(scores)

    embeddings_eval = embeddings_candidate.copy()
    faiss.normalize_L2(embeddings_eval)
    index = faiss.IndexFlatIP(embeddings_candidate.shape[1])
    index.add(embeddings_eval)

    hits = 0
    for i in range(min(50, len(chunks))):
        vec = embeddings_candidate[i:i+1].copy()
        faiss.normalize_L2(vec)
        distances, indices = index.search(vec, 5)
        if i in indices[0]:
            hits += 1

    recall_at_5 = hits / min(50, len(chunks))

    attack_reduction = (np.mean(scores_raw) - rouge_mean) / np.mean(scores_raw) * 100

    results_list.append({
        "n_components": n_components,
        "rouge_mean": rouge_mean,
        "recall_at_5": recall_at_5,
        "attack_reduction": attack_reduction
    })

    print(f"  ROUGE: {rouge_mean:.3f} | Recall@5: {recall_at_5:.1%} | 방어율: {attack_reduction:.1f}%\n")

results_df = pd.DataFrame(results_list)
results_df.to_csv("results.csv", index=False)
print(f"✅ 결과 저장: results.csv")
print(results_df.to_string(index=False))

### 6-3. 최적 n_components 선택 및 트레이드오프 시각화

**선택 기준**:
1. `Recall@5 ≥ 0.90` (RAG 실용 성능 유지)
2. 위 조건 내에서 `attack_reduction` 최대 (방어율 극대화)

**인사이트**: 이 선택 기준은 '보안 강화를 위해 RAG 성능의 최대 10%만 희생한다'는
실용적 가정에 기반한다. 기관의 보안 정책에 따라 임계값을 조정할 수 있다.

출력 그래프:
- 좌: n_components vs 공격 성공률 감소 (낮은 n → 높은 방어)
- 우: n_components vs Recall@5 (낮은 n → 낮은 RAG 성능)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

results_df = pd.read_csv("results.csv")

print("="*80)
print("📊 최적 PCA n_components 찾기")
print("="*80)

filtered_df = results_df[results_df['recall_at_5'] >= 0.90].copy()

if len(filtered_df) > 0:
    optimal_row = filtered_df.loc[filtered_df['attack_reduction'].idxmax()]
    optimal_n = int(optimal_row['n_components'])

    print(f"\n✅ Recall@5 >= 0.90 조건 만족 후보: {len(filtered_df)}개")
    print(f"   최적 선택 (최대 방어율): n={optimal_n}")
    print(f"   - ROUGE: {optimal_row['rouge_mean']:.3f}")
    print(f"   - Recall@5: {optimal_row['recall_at_5']:.1%}")
    print(f"   - 방어율: {optimal_row['attack_reduction']:.1f}%")
else:
    print(f"\n⚠️  Recall@5 >= 0.90 조건 만족 후보 없음")
    best_row = results_df.loc[results_df['recall_at_5'].idxmax()]
    print(f"   최고 RAG 성능: n={int(best_row['n_components'])}")
    print(f"   - Recall@5: {best_row['recall_at_5']:.1%}")

print(f"\n📈 전체 후보 순위:")
print(results_df.sort_values('attack_reduction', ascending=False).to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(results_df['n_components'], results_df['attack_reduction'], 'o-', color='red', linewidth=2, markersize=8)
ax1.axhline(y=20, color='green', linestyle='--', label='목표 방어율 20%')
ax1.set_xlabel('n_components', fontsize=12)
ax1.set_ylabel('공격 성공률 감소 (%)', fontsize=12)
ax1.set_title('PCA 압축률 vs 방어 성능', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.invert_xaxis()

ax2.plot(results_df['n_components'], results_df['recall_at_5'], 's-', color='blue', linewidth=2, markersize=8)
ax2.axhline(y=0.90, color='green', linestyle='--', label='목표 Recall@5 90%')
ax2.set_xlabel('n_components', fontsize=12)
ax2.set_ylabel('Recall@5', fontsize=12)
ax2.set_title('PCA 압축률 vs RAG 성능', fontsize=13, fontweight='bold')
ax2.set_ylim([0.8, 1.05])
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.invert_xaxis()

plt.tight_layout()
plt.savefig('pca_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ 트레이드오프 커브 저장: pca_tradeoff.png")
print("="*80)

### [Feasibility Check] 공용 PCA 벡터 추출 가능성 검증

**가정**: 금융 도메인 임베딩 공간의 주요 방향(PCA components)은 기관/문서 종류와 무관하게 공유된다.

현재 데이터를 절반으로 나눠 '기관 A'로 PCA를 학습하고, '기관 B' 문서에 적용했을 때 성능이 유지되는지 확인한다.

| 지표 | 기준 |
|------|------|
| Recall@5 | ≥ 0.90 (RAG 성능 유지) |
| ROUGE-1 감소 | > 0 (역전공격 방어 유효) |

In [ ]:
import numpy as np
import faiss
import vec2text
import torch
from sklearn.decomposition import PCA
from rouge_score import rouge_scorer
from tqdm import tqdm

# ── 0. 설정 ──────────────────────────────────────────────────────────────────
N_COMPONENTS  = 128   # 공용 PCA 차원 (Cell 29의 최적값으로 교체 가능)
N_ATTACK      = 10    # Vec2Text 공격 샘플 수
K_RECALL      = 5     # Recall@K
N_RECALL_EVAL = 50    # Recall 평가 샘플 수

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')

# ── 1. 원본 임베딩 로드 및 Train/Test 분할 ──────────────────────────────────
embeddings_raw = np.load('embeddings_unsafe_raw.npy')
n_total = len(embeddings_raw)
split   = n_total // 2

train_emb = embeddings_raw[:split]   # '기관 A' — PCA 학습용
test_emb  = embeddings_raw[split:]   # '기관 B' — 공용 PCA 적용 대상
test_chunks = chunks[split:]         # 대응하는 청크

print(f'전체: {n_total}개  |  Train(기관A): {len(train_emb)}개  |  Test(기관B): {len(test_emb)}개')

# ── 2. 기관 A 코퍼스로 공용 PCA 학습 ────────────────────────────────────────
pca = PCA(n_components=N_COMPONENTS, svd_solver='full')
pca.fit(train_emb)

# 공용 배포 아티팩트 저장
np.save('shared_pca_components.npy', pca.components_.astype(np.float32))
np.save('shared_pca_mean.npy',       pca.mean_.astype(np.float32))

variance_retained = pca.explained_variance_ratio_.sum()
print(f'\n[공용 PCA 학습 완료]')
print(f'  components shape : {pca.components_.shape}  ({pca.components_.nbytes / 1e6:.1f} MB)')
print(f'  mean shape       : {pca.mean_.shape}')
print(f'  분산 보존율      : {variance_retained:.1%}')

# ── 3. 공용 PCA 적용 함수 (fit 없이 행렬곱만) ───────────────────────────────
def apply_shared_pca(embeddings: np.ndarray,
                     components: np.ndarray,
                     mean: np.ndarray) -> np.ndarray:
    centered      = embeddings - mean
    compressed    = centered @ components.T       # (N, k)
    reconstructed = compressed @ components + mean # (N, 1536)
    norms = np.linalg.norm(reconstructed, axis=1, keepdims=True)
    return (reconstructed / norms).astype(np.float32)

components = np.load('shared_pca_components.npy')
mean       = np.load('shared_pca_mean.npy')

# 기관 B 문서에 공용 PCA 적용
test_defended = apply_shared_pca(test_emb, components, mean)
print(f'\n[기관 B 방어 벡터] shape: {test_defended.shape}')

# ── 4. Recall@5 평가 ─────────────────────────────────────────────────────────
def eval_recall(embeddings: np.ndarray, k: int = 5, n_eval: int = 50) -> float:
    n_eval = min(n_eval, len(embeddings))
    emb = embeddings[:n_eval].copy()
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    hits = 0
    for i in range(n_eval):
        q = emb[i:i+1].copy()
        _, ids = index.search(q, k)
        if i in ids[0]:
            hits += 1
    return hits / n_eval

recall_raw      = eval_recall(test_emb,      K_RECALL, N_RECALL_EVAL)
recall_defended = eval_recall(test_defended, K_RECALL, N_RECALL_EVAL)

print(f'\n[Recall@{K_RECALL} — 기관 B 문서]')
print(f'  원본 벡터    : {recall_raw:.1%}')
print(f'  공용PCA 방어 : {recall_defended:.1%}  ({"✅ 기준 충족" if recall_defended >= 0.90 else "⚠️  기준 미달 (< 90%)"})')

# ── 5. Vec2Text 역전공격 평가 ────────────────────────────────────────────────
corrector = vec2text.load_pretrained_corrector('text-embedding-ada-002')
scorer_rouge = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

n_atk = min(N_ATTACK, len(test_emb))
originals_test = [test_chunks[i].page_content for i in range(n_atk)]

def attack_rouge(embeddings_tensor):
    scores = []
    for i in tqdm(range(n_atk), desc='Vec2Text 공격'):
        vec = embeddings_tensor[i:i+1]
        result = vec2text.invert_embeddings(
            embeddings=vec,
            corrector=corrector,
            num_steps=20,
            sequence_beam_width=0,
        )
        rouge = scorer_rouge.score(originals_test[i], result[0])['rouge1'].fmeasure
        scores.append(rouge)
    return scores

print('\n[Vec2Text 공격 — 원본 벡터]')
raw_tensor      = torch.tensor(test_emb[:n_atk]).to(device)
scores_raw_test = attack_rouge(raw_tensor)

print('\n[Vec2Text 공격 — 공용PCA 방어 벡터]')
defended_tensor      = torch.tensor(test_defended[:n_atk]).to(device)
scores_defended_test = attack_rouge(defended_tensor)

rouge_raw      = np.mean(scores_raw_test)
rouge_defended = np.mean(scores_defended_test)
attack_reduction = (rouge_raw - rouge_defended) / rouge_raw * 100

# ── 6. Feasibility 최종 판정 ─────────────────────────────────────────────────
print('\n' + '='*70)
print('📋 [Feasibility Check] 공용 PCA 벡터 적용 결과')
print('='*70)
print(f'  학습 기관(A) 샘플 수   : {len(train_emb)}')
print(f'  적용 기관(B) 샘플 수   : {len(test_emb)}')
print(f'  공용 PCA n_components  : {N_COMPONENTS}')
print(f'  배포 아티팩트 크기     : {(pca.components_.nbytes + pca.mean_.nbytes) / 1e6:.2f} MB')
print()
print(f'  Recall@{K_RECALL}  원본 → 방어  : {recall_raw:.1%} → {recall_defended:.1%}')
print(f'  ROUGE-1  원본 → 방어  : {rouge_raw:.3f} → {rouge_defended:.3f}')
print(f'  공격 성공률 감소       : {attack_reduction:.1f}%')
print()

feasible = (recall_defended >= 0.90) and (attack_reduction > 0)
if feasible:
    print('✅ FEASIBLE — 공용 PCA 벡터로 타 기관 문서 방어 가능')
    print('   → shared_pca_components.npy / shared_pca_mean.npy 배포 권장')
elif recall_defended < 0.90:
    print('⚠️  RAG 성능 기준 미달 — n_components 증가 필요')
    print(f'   현재 {N_COMPONENTS} → 256 또는 384 시도 권장')
else:
    print('⚠️  방어 효과 없음 — 코퍼스 다양성 부족 가능성')
print('='*70)
